# LogiScan Stage 1 Retraining (Binary Gatekeeper)

This notebook trains the **Stage 1 Gatekeeper**, which determines if a text contains a logical claim or is just a factual/conversational snippet.

### Step 1: Install Dependencies

In [ ]:
!pip install transformers[torch] datasets accelerate scikit-learn tqdm

### Step 2: Setup Training Logic

In [ ]:
import json

import torch
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ArgumentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }

def load_data(path):
    with open(path) as f:
        raw_data = json.load(f)

    texts, labels = [], []

    # 1. Identified Negative Classes (Factual/Valid)
    negative_classes = ["factual_statement", "valid_reasoning", "none"]

    # 2. Extract structured samples
    for item in raw_data:
        text = item.get("text", "")
        fallacy = item.get("fallacy", "unknown")

        if text and len(text.split()) >= 5:
            texts.append(text)
            # Binary: 0 if identified negative class, 1 otherwise (fallacy)
            if fallacy in negative_classes:
                labels.append(0)
            else:
                labels.append(1)

    # 3. Hard Negative Generation (Fragments of arguments)
    # This prevents the model from just learning 'sentence length' or 'complexity'
    neg_fragments = []
    positives = [t for t, l in zip(texts, labels) if l == 1]
    for t in positives[:1000]:
        words = t.split()
        if len(words) > 8:
            # First few words of a fallacy are often not the fallacy itself
            neg_fragments.append((" ".join(words[:4]), 0))

    texts_final = texts + [t for t, l in neg_fragments]
    labels_final = labels + [l for t, l in neg_fragments]

    return train_test_split(
        texts_final, labels_final, test_size=0.2, random_state=42, stratify=labels_final
    )

### Step 3: Run Training

In [ ]:
DATA_PATH = "unified_training_data.json"
X_train, X_val, y_train, y_val = load_data(DATA_PATH)

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(DEVICE)

train_ds = ArgumentDataset(X_train, y_train, tokenizer)
val_ds = ArgumentDataset(X_val, y_val, tokenizer)

train_loader = DataLoader(train_ds, batch_size=32 if torch.cuda.is_available() else 4, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64 if torch.cuda.is_available() else 8)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
EPOCHS = 2
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps)

best_f1 = 0
for epoch in range(EPOCHS):
    model.train()
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        optimizer.zero_grad()
        out = model(batch["input_ids"].to(DEVICE), attention_mask=batch["attention_mask"].to(DEVICE), labels=batch["label"].to(DEVICE))
        out.loss.backward()
        optimizer.step()
        scheduler.step()

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            logits = model(batch["input_ids"].to(DEVICE), attention_mask=batch["attention_mask"].to(DEVICE)).logits
            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_labels.extend(batch["label"].numpy())

    f1 = f1_score(all_labels, all_preds, average="binary")
    print(f"Val F1: {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        model.save_pretrained("stage1_gatekeeper")
        tokenizer.save_pretrained("stage1_gatekeeper")
        print("✅ Saved Best Model")

### Step 4: Zip and Download

In [ ]:
!zip -r stage1_model.zip stage1_gatekeeper